# LangSmith trace review with ReactCode and Label Studio

This notebook is the companion for the **LangSmith → Label Studio Enterprise** ReactCode integration tutorial.

You will:
- generate or pull traces from LangSmith
- normalize them into a consistent trace-review schema
- create a Label Studio project via API
- import traces as tasks
- review + annotate in a custom ReactCode UI

> **Requirement:** ReactCode is available in **Label Studio Enterprise** only.

## 1) Configuration

Create a `.env` file in the repository root (or the same directory as this notebook) with the following variables:

```bash
# Label Studio Enterprise
LABEL_STUDIO_HOST=http://localhost:8080       # or your LS Enterprise instance URL
LABEL_STUDIO_API_KEY=your_label_studio_api_key

# LangSmith
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_PROJECT=your_project_name           # project name for tracing and fetching
LANGSMITH_ENDPOINT=https://api.smith.langchain.com

# OpenAI (only needed for Section 3a sample trace generation)
OPENAI_API_KEY=your_openai_api_key
```

In [5]:
import os
from dotenv import load_dotenv

# Load .env from current directory or repository root
# override=True ensures kernel picks up .env changes without restart
load_dotenv(override=True)
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), '.env'), override=True)  # fallback: repo root

# Label Studio Enterprise
LABEL_STUDIO_HOST = os.getenv('LABEL_STUDIO_HOST', 'http://localhost:8080')
LABEL_STUDIO_API_KEY = os.getenv('LABEL_STUDIO_API_KEY', '')

# LangSmith
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY', '')
LANGSMITH_PROJECT = os.getenv('LANGSMITH_PROJECT', '')
LANGSMITH_ENDPOINT = os.getenv('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')

# Enable LangSmith tracing for LangChain/LangGraph (auto-instrumentation)
os.environ['LANGSMITH_TRACING'] = 'true'

# OpenAI (only needed for sample trace generation in Section 3a)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')

print('LABEL_STUDIO_HOST:', LABEL_STUDIO_HOST)
print('LANGSMITH_PROJECT:', LANGSMITH_PROJECT or '(not set)')
print('LANGSMITH_ENDPOINT:', LANGSMITH_ENDPOINT)
print('Has LABEL_STUDIO_API_KEY?', bool(LABEL_STUDIO_API_KEY))
print('Has LANGSMITH_API_KEY?', bool(LANGSMITH_API_KEY))
print('Has OPENAI_API_KEY?', bool(OPENAI_API_KEY))

LABEL_STUDIO_HOST: https://app.humansignal.com
LANGSMITH_PROJECT: ls-project
LANGSMITH_ENDPOINT: https://api.smith.langchain.com
Has LABEL_STUDIO_API_KEY? True
Has LANGSMITH_API_KEY? True
Has OPENAI_API_KEY? True


## 2) Install dependencies

In [6]:
!pip -q install requests label-studio-sdk python-dotenv langsmith langchain langchain-openai


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 3) Label Studio ReactCode config
3-panel UI (turn list → detail → annotation) for trace review. Loaded from `label_config.py` in this directory.

In [3]:
# Load 3-panel ReactCode config (label_config.py + template.js in same directory)
_config_paths = [
  'label_config.py',
  os.path.join(os.getcwd(), 'tutorials', 'how-to-review-langsmith-traces-with-label-studio', 'label_config.py'),
]
_config_path = next((p for p in _config_paths if os.path.exists(p)), None)
if _config_path:
  import importlib.util
  _spec = importlib.util.spec_from_file_location('label_config', _config_path)
  _mod = importlib.util.module_from_spec(_spec)
  _spec.loader.exec_module(_mod)
  LABEL_CONFIG_XML = _mod.LABEL_CONFIG_XML
else:
  raise FileNotFoundError('label_config.py not found. Ensure it is in the same directory as this notebook.')
print(LABEL_CONFIG_XML[:300] + '\n...')

<View>
  <ReactCode style="height: 95vh" name="trace" toName="trace" outputs='{"trace_id":"string","turn_id":"string","turn_role":"string","verdict":"string","failure_modes":"array","severity":"string","expected_behavior":"string","comments":"string"}'>
    <![CDATA[
    function TraceAnnotator({ Re
...


## 3a) Generate sample traces (optional)

If you already have traces in LangSmith, **skip this cell** — set `GENERATE_TRACES = False` and go directly to Section 4.

Otherwise, this cell creates a ReAct agent with multiple tools and runs 4 multi-turn conversations to produce realistic traces in your LangSmith project. Requires `OPENAI_API_KEY`.

LangSmith auto-instruments all LangChain/LangGraph calls when `LANGSMITH_TRACING=true` is set — no callback handler needed.

In [4]:
GENERATE_TRACES = True  # Set to False if you already have traces in LangSmith

if GENERATE_TRACES:
    from langsmith import traceable
    from langchain_core.tools import tool
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI
    from langchain.agents import create_agent

    if not OPENAI_API_KEY:
        raise RuntimeError('OPENAI_API_KEY is required for trace generation. Set it in your .env or set GENERATE_TRACES=False.')
    if not LANGSMITH_PROJECT:
        raise RuntimeError('LANGSMITH_PROJECT is required. Set it in your .env file.')

    # --- Tools ---
    @tool
    def calculator(expression: str) -> str:
        """Evaluate a math expression. Input should be a valid Python math expression."""
        try:
            return str(eval(expression))
        except Exception as e:
            return f"Error: {e}"

    @tool
    def search_knowledge_base(query: str) -> str:
        """Search an internal knowledge base for information about company policies, products, or procedures."""
        kb = {
            "refund": "Refund policy: Full refund within 30 days of purchase. After 30 days, store credit only. Damaged items: full refund at any time with photo evidence.",
            "shipping": "Shipping: Standard (5-7 business days, free over $50), Express (2-3 days, $12.99), Overnight ($24.99). International shipping available to 40+ countries.",
            "warranty": "Warranty: 1-year limited warranty on all electronics. 2-year extended warranty available for $29.99. Covers manufacturing defects only.",
            "pricing": "Enterprise pricing: Base plan $99/mo (up to 10 users), Pro plan $249/mo (up to 50 users), Enterprise plan custom pricing. Annual billing saves 20%.",
        }
        query_lower = query.lower()
        results = [v for k, v in kb.items() if k in query_lower]
        return results[0] if results else f"No results found for: {query}"

    @tool
    def get_weather(city: str) -> str:
        """Get the current weather for a city."""
        weather_data = {
            "new york": "New York: 72°F, Partly Cloudy, Humidity 65%, Wind 8 mph SW",
            "london": "London: 58°F, Overcast, Humidity 80%, Wind 12 mph W",
            "tokyo": "Tokyo: 82°F, Clear, Humidity 55%, Wind 5 mph NE",
            "paris": "Paris: 63°F, Light Rain, Humidity 75%, Wind 10 mph NW",
        }
        return weather_data.get(city.lower(), f"Weather data not available for {city}")

    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    agent = create_agent(llm, [calculator, search_knowledge_base, get_weather])

    # --- 4 multi-turn conversations ---
    # LangSmith auto-traces all LangChain/LangGraph calls via env vars.
    # We use @traceable to group multi-turn conversations under a single trace.
    conversations = [
        # 1: Customer support with knowledge base lookups
        [
            "What's the refund policy for a product I bought 3 weeks ago?",
            "What if the item arrived damaged? Does that change anything?",
        ],
        # 2: Multi-step calculation with follow-up
        [
            "I need to calculate the total cost for our team: 35 users on the Pro plan with annual billing. What's the yearly cost?",
            "How much would we save compared to monthly billing?",
        ],
        # 3: Weather + planning across cities
        [
            "I'm planning a trip. What's the weather like in Tokyo and Paris right now?",
            "Based on the weather, which city would be better for outdoor sightseeing today?",
        ],
        # 4: Mixed tool usage — knowledge base + calculator
        [
            "What shipping options do you have, and how much would express shipping cost for 5 orders?",
            "If I spend $250 total on products, which shipping is free? And what's the total with express for the remaining orders?",
        ],
    ]

    for i, conv_messages in enumerate(conversations, 1):
        print(f"\n--- Conversation {i} ---")

        @traceable(name=f"conversation_{i}", run_type="chain")
        def run_conversation(messages):
            chat_history = []
            final_reply = ""
            for msg_text in messages:
                print(f"  User: {msg_text[:80]}...")
                chat_history.append(HumanMessage(content=msg_text))
                result = agent.invoke({'messages': chat_history})
                chat_history = result['messages']
                final_reply = result['messages'][-1].content
                print(f"  Assistant: {final_reply[:100]}...")
            return final_reply

        run_conversation(conv_messages)

    import time
    time.sleep(5)  # Allow LangSmith to ingest traces
    print(f'\n✓ Generated {len(conversations)} traces (1 per conversation). Proceed to Section 4.')
else:
    print('Skipped trace generation. Proceed to Section 4.')


--- Conversation 1 ---
  User: What's the refund policy for a product I bought 3 weeks ago?...
  Assistant: The refund policy states that you can receive a full refund within 30 days of purchase. Since you bo...
  User: What if the item arrived damaged? Does that change anything?...
  Assistant: If the item arrived damaged, you can still receive a full refund at any time, as long as you provide...

--- Conversation 2 ---
  User: I need to calculate the total cost for our team: 35 users on the Pro plan with a...
  Assistant: The total yearly cost for 35 users on the Pro plan with annual billing is $2,390.40....
  User: How much would we save compared to monthly billing?...
  Assistant: You would save approximately $597.60 by choosing annual billing over monthly billing for the Pro pla...

--- Conversation 3 ---
  User: I'm planning a trip. What's the weather like in Tokyo and Paris right now?...
  Assistant: The current weather is as follows:

- **Tokyo**: 82°F, Clear, Humidity 55%, Wi

## 4) LangSmith client

Uses the `langsmith` SDK to fetch traces (runs) from your project. Runs are hierarchical — each trace is a tree of runs sharing the same `trace_id`.

In [7]:
from langsmith import Client
from typing import Any, Dict, List

if not LANGSMITH_API_KEY:
    raise RuntimeError('Missing LANGSMITH_API_KEY — set it in your .env file.')
if not LANGSMITH_PROJECT:
    raise RuntimeError('Missing LANGSMITH_PROJECT — set it in your .env file.')

ls_client = Client(api_key=LANGSMITH_API_KEY, api_url=LANGSMITH_ENDPOINT)
print(f'LangSmith client ready — project: {LANGSMITH_PROJECT}')

LangSmith client ready — project: ls-project


## 5) Normalize LangSmith traces → unified schema

Converts LangSmith runs into a flat list of turns (user, assistant, tool) that the ReactCode template can render. Each turn has a `role`, `content`, and optional `tool_calls`, `usage`, and `model`.

In [8]:
import json as _json


def _to_str(x):
    """Safely convert any value to a readable string."""
    if x is None:
        return ''
    if isinstance(x, str):
        return x
    if isinstance(x, (dict, list)):
        try:
            return _json.dumps(x, indent=2, default=str)
        except Exception:
            return str(x)
    return str(x)


def _extract_content(obj):
    """Extract plain-text content from various data shapes."""
    if obj is None:
        return ''
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        for key in ('content', 'text', 'input', 'output', 'result'):
            if isinstance(obj.get(key), str) and obj[key].strip():
                return obj[key]
        return _to_str(obj)
    if isinstance(obj, list):
        parts = [_extract_content(item) for item in obj if _extract_content(item).strip()]
        return '\n'.join(parts) if parts else _to_str(obj)
    return str(obj)


def normalize_langsmith_trace(root_run, all_runs):
    """Convert LangSmith runs into the unified trace schema.

    LangSmith run hierarchy (from LangGraph + auto-instrumentation):
      conversation_N (root, run_type=chain)
        └── LangGraph (run_type=chain)
              ├── agent (run_type=chain)
              │     └── ChatOpenAI (run_type=llm)
              ├── tools (run_type=chain)
              │     └── search_knowledge_base (run_type=tool)
              ├── agent (run_type=chain)
              │     └── ChatOpenAI (run_type=llm)
              ...

    Strategy:
      1. Sort runs chronologically
      2. run_type=llm → extract user messages from inputs + assistant response from outputs
      3. run_type=tool → extract tool execution as a tool turn
      4. Skip structural runs (run_type=chain) that are just wrappers
    """
    trace_id = str(root_run.get('trace_id') or root_run.get('id', ''))
    session_id = str(root_run.get('session_id') or trace_id)

    runs_sorted = sorted(all_runs, key=lambda r: str(r.get('start_time') or r.get('dotted_order') or ''))

    turns = []
    turn_counter = 0
    seen_user_messages = set()

    def add_turn(role, content, **kwargs):
        nonlocal turn_counter
        if not content or not content.strip():
            return
        turn = {
            'turn_id': f'turn_{turn_counter}',
            'role': role,
            'content': content.strip(),
            'timestamp': kwargs.get('timestamp', ''),
        }
        for k in ('model', 'usage', 'tool_calls', 'tool_name', 'tool_input'):
            if k in kwargs and kwargs[k]:
                turn[k] = kwargs[k]
        turns.append(turn)
        turn_counter += 1

    for run in runs_sorted:
        run_type = (run.get('run_type') or '').lower()
        ts = str(run.get('start_time') or '')
        inp = run.get('inputs') or {}
        out = run.get('outputs') or {}

        # ── LLM run: ChatOpenAI ──
        # inputs: {messages: [[{type: "human", content: "..."}]]} or {messages: [{role, content}]}
        # outputs: {generations: [[{message: {content, tool_calls}}]]} or {output: {content}}
        if run_type == 'llm':
            # Extract user messages from inputs
            messages = inp.get('messages') or inp.get('input') or []
            # LangSmith may nest messages as [[msg1, msg2, ...]] or [msg1, msg2, ...]
            if isinstance(messages, list) and messages and isinstance(messages[0], list):
                messages = messages[0]  # unwrap outer list

            if isinstance(messages, list):
                for msg in messages:
                    if isinstance(msg, dict):
                        # LangSmith serializes LangChain messages with kwargs wrapper
                        if 'kwargs' in msg:
                            role = msg.get('id', [''])[-1] if isinstance(msg.get('id'), list) else ''
                            msg_data = msg.get('kwargs', {})
                        else:
                            role = msg.get('role') or msg.get('type', '')
                            msg_data = msg

                        if role in ('user', 'human', 'HumanMessage'):
                            content = msg_data.get('content', '')
                            if isinstance(content, list):
                                content = ' '.join(
                                    p.get('text', '') if isinstance(p, dict) else str(p) for p in content
                                )
                            if content and content.strip():
                                msg_key = content[:200]
                                if msg_key not in seen_user_messages:
                                    seen_user_messages.add(msg_key)
                                    add_turn('user', content, timestamp=ts)

            # Extract assistant response from outputs
            assistant_content = ''
            tool_calls = []

            # Shape 1: {generations: [[{message: {content, additional_kwargs: {tool_calls}}}]]}
            gens = out.get('generations')
            if isinstance(gens, list) and gens and isinstance(gens[0], list) and gens[0]:
                gen = gens[0][0]
                if isinstance(gen, dict):
                    message = gen.get('message', {})
                    if 'kwargs' in message:
                        msg_data = message.get('kwargs', {})
                    else:
                        msg_data = message
                    assistant_content = msg_data.get('content', '') or ''

                    ak = msg_data.get('additional_kwargs') or {}
                    tc_raw = ak.get('tool_calls') or msg_data.get('tool_calls') or []
                    for tc in tc_raw:
                        if isinstance(tc, dict):
                            func = tc.get('function') or {}
                            tool_calls.append({
                                'tool_name': func.get('name') or tc.get('name', 'unknown'),
                                'input': _to_str(func.get('arguments') or tc.get('args', '')),
                                'call_id': tc.get('id', ''),
                            })

            # Shape 2: {output: {content: "..."}}
            if not assistant_content and isinstance(out.get('output'), dict):
                assistant_content = out['output'].get('content', '') or ''

            # Shape 3: direct {content: "..."}
            if not assistant_content and isinstance(out.get('content'), str):
                assistant_content = out['content']

            model_name = run.get('name') or ''
            usage = None
            prompt_tokens = run.get('prompt_tokens') or 0
            completion_tokens = run.get('completion_tokens') or 0
            if prompt_tokens or completion_tokens:
                usage = {
                    'input_tokens': prompt_tokens,
                    'output_tokens': completion_tokens,
                }

            if assistant_content and assistant_content.strip():
                add_turn(
                    'assistant', assistant_content,
                    timestamp=ts, model=model_name,
                    usage=usage,
                    tool_calls=tool_calls if tool_calls else None,
                )
            elif tool_calls:
                tool_names = ', '.join(tc['tool_name'] for tc in tool_calls)
                add_turn(
                    'assistant', f'[Calling: {tool_names}]',
                    timestamp=ts, model=model_name,
                    usage=usage,
                    tool_calls=tool_calls,
                )

        # ── Tool run ──
        # inputs: {input: "query"} or tool-specific args
        # outputs: {output: "result"}
        elif run_type == 'tool':
            tool_name = run.get('name') or 'unknown'
            tool_input = ''
            if isinstance(inp, dict):
                tool_input = _to_str(inp.get('input') or inp.get('query') or inp)
            else:
                tool_input = _to_str(inp)

            tool_output = ''
            if isinstance(out, dict):
                tool_output = out.get('output', '') or _extract_content(out)
            elif out:
                tool_output = _extract_content(out)

            if tool_output:
                add_turn(
                    'tool', str(tool_output),
                    timestamp=ts,
                    tool_name=tool_name,
                    tool_input=tool_input,
                )

    # Fallback: if no turns extracted, use root run inputs/outputs
    if not turns:
        root_inputs = root_run.get('inputs') or {}
        root_outputs = root_run.get('outputs') or {}
        root_input = _extract_content(root_inputs.get('input') or root_inputs)
        root_output = _extract_content(root_outputs.get('output') or root_outputs)
        if root_input:
            add_turn('user', root_input, timestamp=str(root_run.get('start_time', '')))
        if root_output:
            add_turn('assistant', root_output, timestamp=str(root_run.get('end_time', '')))

    return {
        'trace_id': trace_id,
        'session_id': session_id,
        'metadata': {
            'name': root_run.get('name') or '',
            'source': 'langsmith',
            'tags': root_run.get('tags') or [],
            'start_time': str(root_run.get('start_time') or ''),
        },
        'turns': turns,
    }


print('✓ Normalization functions defined')

✓ Normalization functions defined


## 6) Fetch, normalize, and import into Label Studio

Fetches traces from LangSmith, normalizes them, creates a Label Studio project with the ReactCode config, and imports the tasks.

In [9]:
from label_studio_sdk import LabelStudio
from label_studio_sdk.core.request_options import RequestOptions
from typing import Any, Dict, List

# Extended timeout for large ReactCode label configs
_REQUEST_OPTS = RequestOptions(timeout_in_seconds=120)


def create_project(ls_host: str, api_key: str, title: str, label_config: str) -> int:
    client = LabelStudio(base_url=ls_host, api_key=api_key)
    project = client.projects.create(title=title, label_config=label_config, request_options=_REQUEST_OPTS)
    return int(project.id)


def import_tasks(ls_host: str, api_key: str, project_id: int, tasks: List[Dict[str, Any]]) -> Any:
    client = LabelStudio(base_url=ls_host, api_key=api_key)
    return client.projects.import_tasks(id=project_id, request=tasks, return_task_ids=True)


if not LABEL_STUDIO_API_KEY:
    raise RuntimeError('Missing LABEL_STUDIO_API_KEY — set it in your .env file.')

# 1) Fetch root runs (traces) from LangSmith
print(f'Fetching traces from LangSmith project: {LANGSMITH_PROJECT}...')
root_runs_iter = ls_client.list_runs(
    project_name=LANGSMITH_PROJECT,
    is_root=True,
    limit=20,
)
root_runs = [r.model_dump() for r in root_runs_iter]

if not root_runs:
    raise RuntimeError(
        'No traces returned from LangSmith. Ensure you have traces in your project '
        'and that LANGSMITH_* credentials are correct. Run Section 3a to generate sample traces.'
    )

print(f'Fetched {len(root_runs)} root runs from LangSmith')

# 2) For each root run, fetch all child runs and normalize
tasks: List[Dict[str, Any]] = []
skipped = 0
for root in root_runs:
    trace_id = str(root.get('trace_id') or root.get('id'))
    if not trace_id:
        continue

    # Fetch all runs in this trace
    trace_runs_iter = ls_client.list_runs(
        project_name=LANGSMITH_PROJECT,
        trace_id=trace_id,
    )
    all_runs = [r.model_dump() for r in trace_runs_iter]

    # Skip traces with only the root run (no child instrumentation)
    if len(all_runs) <= 1:
        skipped += 1
        continue

    normalized = normalize_langsmith_trace(root, all_runs)

    if normalized['turns']:
        tasks.append({'data': normalized})
        n_user = sum(1 for t in normalized['turns'] if t['role'] == 'user')
        n_asst = sum(1 for t in normalized['turns'] if t['role'] == 'assistant')
        n_tool = sum(1 for t in normalized['turns'] if t['role'] == 'tool')
        print(f"  + Trace {trace_id[:12]}... -> {len(normalized['turns'])} turns "
              f"({n_user} user, {n_asst} assistant, {n_tool} tool)")

if skipped:
    print(f'  (skipped {skipped} traces without child runs)')

print(f'\nPrepared {len(tasks)} tasks for import')

# 3) Create project and import
project_id = create_project(
    ls_host=LABEL_STUDIO_HOST,
    api_key=LABEL_STUDIO_API_KEY,
    title=f'LangSmith Trace Review ({LANGSMITH_PROJECT})',
    label_config=LABEL_CONFIG_XML,
)
print(f'Created project: {project_id}')

resp = import_tasks(LABEL_STUDIO_HOST, LABEL_STUDIO_API_KEY, project_id, tasks)
print(f'Imported {len(tasks)} tasks')

print(f'\nDone! Open your project: {LABEL_STUDIO_HOST.rstrip("/")}/projects/{project_id}')

Fetching traces from LangSmith project: ls-project...
Fetched 4 root runs from LangSmith
  + Trace 019cc2e6-4f4... -> 6 turns (2 user, 3 assistant, 1 tool)
  + Trace 019cc2e6-3cf... -> 7 turns (2 user, 3 assistant, 2 tool)
  + Trace 019cc2e6-2a2... -> 10 turns (2 user, 5 assistant, 3 tool)
  + Trace 019cc2e6-1b1... -> 6 turns (2 user, 3 assistant, 1 tool)

Prepared 4 tasks for import
Created project: 236878
Imported 4 tasks

Done! Open your project: https://app.humansignal.com/projects/236878


## What's next

- **Start annotating**: Open the project link above and click through traces in the ReactCode UI
- **Share with SMEs**: Invite domain experts to your Label Studio project for collaborative evaluation
- **Incremental sync**: Re-run sections 4–6 periodically to pull new traces
- **Custom taxonomy**: Edit `template.js` to add failure modes specific to your domain
- **Langfuse / Braintrust**: See companion tutorials for other observability platforms